In [1]:
import arcpy
from arcpy.sa import Raster
import itertools
import pandas as pd
import os
import csv

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "75%"
arcpy.env.scratchWorkspace = "F:/scratch"
workspace = "F:/tc_model_rasters"
arcpy.env.workspace = workspace
#arcpy.env.extent = "-91.819517 36.517373 -86.383202 42.913071" 
arcpy.env.extent = arcpy.Extent(-91.825, 42.917, -91.824, 42.918)
arcpy.env.cellSize = 10000000000

# Check what arcpy thinks the extent is
print("env.extent:", arcpy.env.extent)
print("env.cellSize:", arcpy.env.cellSize)

env.extent: -91.825 42.917 -91.824 42.918 NaN NaN NaN NaN
env.cellSize: 10000000000


In [2]:
# prediction depths
depths = [i for i in range(5,105,10)]

In [3]:
# folder containing model coefficients
model_folder = "C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch6_CASC_Project/Dual-Risk-Repo/NWCA_Model"
model_path = os.path.normpath(os.path.join(model_folder,'Best_TC_Concentration_Model.csv'))

In [4]:
# load best model
model_coeffs = pd.read_csv(model_path)

In [5]:
# set the intercept
intercept = model_coeffs.loc[0,"values"]

In [6]:
# get scalar depth effect
b_depth = model_coeffs.loc[1,"values"]

In [7]:
# list every raster file and its main-effect scalar
raster_coefficients = {}
#for i in range(2,11):
for i in range(2,3):
    key = model_coeffs.loc[i,"labels"] + ".tif"
    raster_coefficients[key] = model_coeffs.loc[i,"values"]
raster_coefficients

{'MAP.tif': np.float64(0.432018763535628)}

In [8]:
# get coefficients for rasters that have interaction with depth
depth_raster_coefficients = {}
for i in range(11,14):
    key = model_coeffs.loc[i,"labels"].split(':')[1] + ".tif"
    depth_raster_coefficients [key] = model_coeffs.loc[i,"values"] 

In [9]:
# list pairs and their interaction scalar
interaction_coefficients = {}
for i in range(14,36):
    keys = model_coeffs.loc[i,"labels"].split(':')
    for j in range(2):
        keys[j] = keys[j] + ".tif"
    keys = tuple(keys)
    interaction_coefficients[keys] = model_coeffs.loc[i,"values"]

In [10]:
print("Loading rasters...")
raster_objects = {name: Raster(name) for name in raster_coefficients}

Loading rasters...


In [11]:
# start with intercept
result = arcpy.sa.Con(raster_objects[list(raster_objects.keys())[0]] >= -9999, intercept)

# Simpler: use Float raster math — initialize from intercept directly
result = intercept  # arcpy.sa handles scalar + Raster arithmetic

In [12]:
# Add main effects: b_i * R_i
print("Adding main effects...")
for name, coef in raster_coefficients.items():
    r = raster_objects[name]
    term = coef * r
    result = result + term
    print(f"  + {coef} * {name}")

Adding main effects...
  + 0.432018763535628 * MAP.tif


In [13]:
# Add interaction terms: b_ij * (R_i * R_j)
#print("Adding interaction terms...")

#for (name_i, name_j), coef in interaction_coefficients.items():
#    if name_i not in raster_objects or name_j not in raster_objects:
#        print(f"  WARNING: Skipping ({name_i}, {name_j}) — raster not found in raster_coefficients")
#        continue
#    r_i = raster_objects[name_i]
#    r_j = raster_objects[name_j]
#    term = coef * (r_i * r_j)
#    result = result + term
#    print(f"  + {coef} * ({name_i} x {name_j})")

In [ ]:
# add depth effects and write to file
for depth in depths:
    print("Adding scalar depth effect")
    #result = result + b_depth*depth

    print("Adding interaction depth effects")
    for name, coef in depth_raster_coefficients.items():
        #r = raster_objects[name]
        #term = coef * depth * r
        print(f"  + {coef} * {depth} * {name}")
    
    # Output raster path (use .tif or a geodatabase path)
    output_raster = f"F:/tc_model_results/tc_{depth}cm_concentration.tif"

    # Save output
    print(f"\nSaving output to: {output_raster}")
    result.save(output_raster)
    print("Done.")
     
arcpy.CheckInExtension("Spatial")

Adding scalar depth effect
Adding interaction depth effects
  + 7.48526288239338e-05 * 5 * MAP.tif
  + -0.0055825798030259 * 5 * MAT.tif
  + -6.46848625299262e-05 * 5 * SOC_Stock.tif

Saving output to: F:/tc_model_results/tc_5cm_concentration.tif
